# AIHub 71363 받기 → 변환 (로컬 Jupyter, 윈도우·맥·리눅스)

- **국내 PC에서 실행하십시오.** AIHub 는 해외 IP(Colab 등) 다운로드를 막습니다.
- 준비물: 71363 데이터셋 **다운로드 승인** + 마이페이지에서 발급한 **API 키** (노트북에 적지 않고 입력창으로 받습니다)
- WSL·bash 없이 파이썬만으로 받습니다 (`scripts/aihub_download.py`). 받은 뒤 분할 조각 병합·압축 해제까지 자동입니다.

| 단계 | 파일 | 크기 | key |
|---|---|---|---|
| `val` (먼저) | VS_02. Skyset (검증 원천) | 5.54 GB | 491174 |
| | VL_01.LABEL_02. Skyset (검증 라벨 TIF) | 16 MB | 491178 |
| `train` | TS_02. Skyset (학습 원천) | 42.91 GB | 491162 |
| | TL_01.LABEL_02. Skyset (학습 라벨 TIF) | 94 MB | 491166 |

디스크 여유: `val` 약 15GB, `train` 약 100GB (조각 병합 때 잠시 두 배가 필요).

In [ ]:
# [1] 설정
import os, sys, subprocess, shutil, zipfile
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))
import aihub_download as ad

STAGE = 'val'                     # 'val' 먼저, 확인 후 'train'
OUT = ROOT / 'aihub_71363'        # 받은 원본이 풀리는 곳 (git 에서 제외됨: 커밋하지 마십시오)
DATA = ROOT / 'data'              # 변환된 학습 쌍이 저장되는 곳
FILES = {'val': ['491174', '491178'], 'train': ['491162', '491166']}
OUT.mkdir(exist_ok=True)
print('저장 위치', OUT, '| 디스크 여유', round(shutil.disk_usage(OUT).free / 2**30, 1), 'GB')

In [ ]:
# [2] API 키 입력 (공백·따옴표 자동 제거, 노트북에 남지 않음)
from getpass import getpass
AIHUB_KEY = os.environ.get('AIHUB_API_KEY') or getpass('AIHub API 키: ')
AIHUB_KEY = AIHUB_KEY.strip().strip("'\"")
print('키 길이', len(AIHUB_KEY))

In [ ]:
# [3] 다운로드 주소 확인 (공식 aihubshell 스크립트에서 읽음)
try:
    shell = ad.fetch_shell(OUT)
    TEMPLATE, how = ad.down_template(shell)
    print(how); print('\n'.join(ad.shell_lines(shell)[:12]))
except Exception as e:
    TEMPLATE, how = ad.DEFAULT_DOWN, f'기본 주소 사용 ({e})'
    print(how)
print('주소:', TEMPLATE)

In [ ]:
# [4] 다운로드 + tar 해제 + 조각 병합 + zip 해제 (조각·zip 은 지워서 디스크 절약)
#     오류 메시지 예) '인증실패' -> 승인·키 확인 / '해외에서의 다운로드를 제한' -> 국내 네트워크에서 실행
ad.run('71363', FILES[STAGE], AIHUB_KEY, OUT, template=TEMPLATE, delete=True)

In [ ]:
# [5] 받은 폴더 찾기 + 원천 형식 확인
def find_dir(pattern):
    c = [p for p in OUT.rglob(pattern) if p.is_dir()]
    return max(c, key=lambda p: sum(1 for _ in p.rglob('*.tif'))) if c else None

SRC = find_dir('VS_02*' if STAGE == 'val' else 'TS_02*')
LAB = find_dir('VL_01*LABEL*' if STAGE == 'val' else 'TL_01*LABEL*')
print('원천', SRC, '| 라벨', LAB)
for name, d in (('원천', SRC), ('라벨', LAB)):
    fs = sorted(d.rglob('*.tif')) if d else []
    print(f'{name} TIF {len(fs)}개', fs[0].name if fs else '')
if SRC:
    try:
        import tifffile
        a = tifffile.imread(str(next(SRC.rglob('*.tif'))))
    except Exception:
        import cv2; a = cv2.imread(str(next(SRC.rglob('*.tif'))), cv2.IMREAD_UNCHANGED)
    print('원천 형식: shape', a.shape, 'dtype', a.dtype, 'min/max', a.min(), a.max())
    print('-> 이 줄을 Claude 에게 알려 주세요 (밴드 순서 확인용)')

In [ ]:
# [6] 변환: 대회 규격 전/후 쌍 (256x256, 0.55~0.7m) + 합성 변화
BANDS = ['0', '1', '2']            # 원천이 4밴드 BGRN 이면 ['2', '1', '0']
dst = DATA / ('aihub_val' if STAGE == 'val' else 'aihub_train')
extra = [] if STAGE == 'val' else ['--windows', '3', '--max-pairs', '1']
cmd = [sys.executable, str(ROOT / 'scripts/prepare_aihub71363.py'), '--images', str(SRC), '--labels', str(LAB),
       '--out', str(dst), '--bands', *BANDS, *extra]
print(' '.join(cmd)); subprocess.run(cmd, check=True, cwd=ROOT)
print('쌍 수', sum(1 for _ in dst.glob('*/pre.png')))

In [ ]:
# [7] 미리보기: 전 | 후 | 정답(빨강 증축, 주황 벌목, 어둡게 = 무시) - 합성이 자연스러운지 눈으로 확인
import cv2, numpy as np, random
from IPython.display import Image as IPImage, display
ds = [d for d in dst.iterdir() if (d / 'pre.png').exists()]
pos = [d for d in ds if cv2.imread(str(d/'building.png'), 0).any() or cv2.imread(str(d/'tree.png'), 0).any()]
rows = []
for d in random.sample(pos, min(4, len(pos))) + random.sample(ds, min(2, len(ds))):
    pre, post = cv2.imread(str(d/'pre.png')), cv2.imread(str(d/'post.png')); ov = post.copy()
    for f, col in (('building.png', (0, 0, 255)), ('tree.png', (0, 165, 255))):
        m = cv2.imread(str(d/f), 0) > 0; ov[m] = (0.45*ov[m] + 0.55*np.array(col)).astype(np.uint8)
    if (d/'ignore.png').exists():
        m = cv2.imread(str(d/'ignore.png'), 0) > 0; ov[m] = (0.5*ov[m]).astype(np.uint8)
    rows.append(np.concatenate([pre, post, ov], 1))
cv2.imwrite(str(DATA / f'preview_{STAGE}.png'), np.concatenate(rows, 0))
display(IPImage(str(DATA / f'preview_{STAGE}.png')))

In [ ]:
# [8] Colab 학습용 zip (Google Drive 의 knps/ 폴더에 올리십시오)
zpath = DATA / f'{dst.name}.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_STORED) as z:   # PNG 는 이미 압축돼 있어 저장만
    for p in dst.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(DATA).as_posix())
print(zpath, round(zpath.stat().st_size / 2**30, 2), 'GB')